In [1]:
import os
import shutil
from sklearn.model_selection import train_test_split

# ---- PATHS ----
DATASET_PATH = "/kaggle/input/datasets/abhishekpriya/vehicle-detection-dataset-multiclass/HeteroTraffic Annotated Dataset for Multi-Class Ve/roadvision_blur/roadvision_blur"  # change this
OUTPUT_PATH  = "/kaggle/working/HeteroTraffic_split"

images_path = os.path.join(DATASET_PATH, "images")
labels_path = os.path.join(DATASET_PATH, "labels")

# ---- CREATE FOLDERS ----
for split in ["train", "val", "test"]:
    os.makedirs(os.path.join(OUTPUT_PATH, "images", split), exist_ok=True)
    os.makedirs(os.path.join(OUTPUT_PATH, "labels", split), exist_ok=True)

# ---- GET FILES ----
images = [f for f in os.listdir(images_path) if f.endswith(".jpg")]

# ---- SPLIT (80/12/8) ----
train, temp = train_test_split(images, test_size=0.20, random_state=42)
val, test   = train_test_split(temp, test_size=0.40, random_state=42)

# ---- COPY ----
def move(files, split):
    for f in files:
        shutil.copy(os.path.join(images_path, f),
                    os.path.join(OUTPUT_PATH, "images", split, f))

        label = f.replace(".jpg", ".txt")
        src_label = os.path.join(labels_path, label)
        if os.path.exists(src_label):
            shutil.copy(src_label,
                        os.path.join(OUTPUT_PATH, "labels", split, label))

move(train, "train")
move(val, "val")
move(test, "test")

print("Done ✅")

Done ✅


In [2]:
with open("/kaggle/working/data.yaml", "w") as f:
    f.write("""path: /kaggle/working/HeteroTraffic_split

train: images/train
val: images/val
test: images/test

names:
  0: Bicycle
  1: Bus
  2: Bhotbhoti
  3: Car
  4: CNG
  5: Easybike
  6: Leguna
  7: Motorbike
  8: MPV
  9: Pedestrian
  10: Pickup
  11: PowerTiller
  12: Rickshaw
  13: ShoppingVan
  14: Truck
  15: Van
  16: Wheelbarrow
""")

In [3]:
# Install dependencies
!pip install ultralytics -q
!pip install roboflow -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 21.8 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.9/175.9 kB 4.7 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 38.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 68.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 98.6 MB/s eta 0:00:00:00:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2026.2.0 which is incompatible.


In [4]:
import os
import yaml
import shutil
from pathlib import Path
from ultralytics import YOLO

# ── Configuration ──────────────────────────────────────────────
# Update this to your Kaggle dataset path
DATASET_PATH = '/kaggle/working/HeteroTraffic_split'   # <-- CHANGE THIS
PROJECT_NAME = 'vehicle_detection'
MODEL_BASE   = 'yolov8m.pt'   # yolov8n / s / m / l / x
EPOCHS       = 80
IMG_SIZE     = 640
BATCH_SIZE   = 16
# ───────────────────────────────────────────────────────────────

print('Ultralytics version:', __import__('ultralytics').__version__)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics version: 8.4.38


In [5]:
# ── Train ───────────────────────────────────────────────────────
data_yaml = '/kaggle/working/data.yaml'

model = YOLO(MODEL_BASE)

results = model.train(
    data    = data_yaml,
    epochs  = EPOCHS,
    imgsz   = IMG_SIZE,
    batch   = BATCH_SIZE,
    project = '/kaggle/working/runs',
    name    = PROJECT_NAME,
    device  = 0,            # GPU
    patience= 15,           # early-stop patience
    save    = True,
    plots   = True,
    verbose = True,
    # Augmentation
    hsv_h   = 0.015,
    hsv_s   = 0.7,
    hsv_v   = 0.4,
    flipud  = 0.0,
    fliplr  = 0.5,
    mosaic  = 1.0,
)

Ultralytics 8.4.38 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=80, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=vehicle_detection, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience

In [8]:
best_weights = f'/kaggle/working/runs/{PROJECT_NAME}/weights/best.pt'
model_best = YOLO(best_weights)
metrics = model_best.val(data=data_yaml, imgsz=IMG_SIZE)
print('mAP50:', metrics.box.map50)
print('mAP50-95:', metrics.box.map)

Ultralytics 8.4.38 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 93 layers, 25,849,603 parameters, 0 gradients, 78.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 3819.2±1111.8 MB/s, size: 340.4 KB)
val: Scanning /kaggle/working/HeteroTraffic_split/labels/val.cache... 2077 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 2077/2077 792.0Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 130/130 3.8it/s 34.7s0.3s
                   all       2077       9633      0.839      0.842      0.889      0.681
               Bicycle        192        383      0.856      0.839      0.893      0.636
                   Bus        243        658      0.884       0.89      0.937      0.761
             Bhotbhoti        111        138      0.899      0.906      0.969      0.792
                   Car        429       1281       0.84      0.881      0.921      0.733
                 

In [9]:
# ── Copy weights to /kaggle/working so you can download them ────
output_path = '/kaggle/working/model1_vehicle_detection.pt'
shutil.copy(best_weights, output_path)
print(f'Model saved to: {output_path}')
print('Download it from the Kaggle Output tab.')

Model saved to: /kaggle/working/model1_vehicle_detection.pt
Download it from the Kaggle Output tab.
